# `food.parquet` — Exploratory Data Analysis (EDA) / Data Profiling

Before deciding how to re-extract / re-clean `products.db`, this notebook looks at the
raw source file `food.parquet` (the global Open Food Facts export, kept under
`raw_fooddata/`) on its own terms first — no assumptions carried in from anywhere else.
How many rows and columns does it have, what are the fields, how complete is each one,
and what does the underlying data structure actually look like?

Every cell here was actually run, and every output shown is the real captured result —
nothing is a fabricated preview. It was run locally
(`Desktop/5120 IE/FOODTRUST DATA/product_data/raw_fooddata/`) with `pyarrow`, without
loading the full 7.3GB file into memory: Parquet is columnar, so only the needed columns
get read, and a lot of the statistics below (e.g. null counts) are pulled straight out of
the file's own metadata (row-group statistics) without scanning the actual data at all —
that's why several cells below finish in a few seconds.

**The questions this notebook is answering**: how many rows/columns does this file have,
what are the fields, what's the missing-value picture, and what does the storage
structure look like (is every field a simple scalar value, or something more complex)?


## 1. Overall shape

Start with the basics: row count, column count, and column names.

In [ ]:
import pyarrow.parquet as pq

pf = pq.ParquetFile("food.parquet")
schema = pf.schema_arrow
print(f"food.parquet: {pf.metadata.num_rows:,} rows, {len(schema.names)} columns")
print()
print("-- all column names --")
for i, name in enumerate(schema.names):
    print(f"{i+1}. {name}")

food.parquet: 4,701,613 rows, 111 columns

-- all column names --
1. additives_n
2. additives_tags
3. allergens_tags
4. brands_tags
5. brands
6. categories
7. categories_tags
8. categories_properties
9. checkers_tags
10. ciqual_food_name_tags
11. cities_tags
12. code
13. compared_to_category
14. complete
15. completeness
16. correctors_tags
17. countries_tags
18. created_t
19. creator
20. data_quality_errors_tags
21. data_quality_info_tags
22. data_quality_warnings_tags
23. data_sources_tags
24. environmental_score_data
25. environmental_score_grade
26. environmental_score_score
27. environmental_score_tags
28. editors
29. emb_codes_tags
30. emb_codes
31. entry_dates_tags
32. food_groups_tags
33. generic_name
34. images
35. informers_tags
36. ingredients_analysis_tags
37. ingredients_from_palm_oil_n
38. ingredients_n
39. ingredients_original_tags
40. ingredients_percent_analysis
41. ingredients_tags
42. ingredients_text
43. ingredients_with_specified_percent_n
44. ingredients_with_unsp

**Note**: these 4,701,613 rows are Open Food Facts' **entire global product catalogue**,
not just baby foods — the 7,036 rows currently in `products.db` are a small subset
filtered from here by `categories_tags` containing `en:baby-foods` (about 0.15% of the
whole file). So every coverage percentage computed below reflects the **global**
dataset, not necessarily the true coverage inside the baby-food subset (the two may
differ — worth verifying separately later).

`categories_tags` is a list field (a row can carry many category tags at once), and it's
the field the `en:baby-foods` filter is based on. Checked directly here: does that filter
already include formula/growth-milk products, or would they need a separate tag added.

In [ ]:
import pyarrow.parquet as pq

pf = pq.ParquetFile("food.parquet")
baby_foods = baby_milks = infant_formulas = both = 0
for batch in pf.iter_batches(columns=["categories_tags"], batch_size=200000):
    for tags in batch.column("categories_tags").to_pylist():
        if not tags:
            continue
        has_bf = "en:baby-foods" in tags
        has_bm = "en:baby-milks" in tags
        has_if = "en:infant-formulas" in tags
        baby_foods += has_bf
        baby_milks += has_bm
        infant_formulas += has_if
        both += has_bf and (has_bm or has_if)

print(f"en:baby-foods: {baby_foods:,}")
print(f"en:baby-milks (formula/growth-milk subcategory): {baby_milks:,}")
print(f"en:infant-formulas (subcategory): {infant_formulas:,}")
print(f"en:baby-foods AND (baby-milks OR infant-formulas): {both:,}")

en:baby-foods: 7,036
en:baby-milks (formula/growth-milk subcategory): 1,356
en:infant-formulas (subcategory): 517
en:baby-foods AND (baby-milks OR infant-formulas): 1,356

`en:baby-foods` already covers all 1,356 `en:baby-milks` rows and all 517
`en:infant-formulas` rows (the AND count equals the `en:baby-milks` count exactly, so
`en:infant-formulas` is itself a subset of `en:baby-milks` here) — the current filter does
not miss formula/growth-milk products; they're part of the 7,036, not excluded by the
extraction step. (Separately, `product_nppm_category_mapping.db` routes these 1,411 rows
to `out_of_scope` for WHO NPPM scoring — a downstream classification choice, not a gap in
this extraction.)

## 2. Missing rate for every scalar field (all 57)

Of the 111 fields, 57 are scalar (one simple value per row, e.g. `brands`, `code`) and
54 are nested (a list or struct per row, e.g. `nutriments`, `images`, `categories_tags`).
This section covers the 57 scalar fields, computed directly from the row-group statistics
(`null_count`) stored in the Parquet file's footer — no need to decompress the actual 4,701,613 rows of data. That's why this
finishes in seconds rather than the tens-of-seconds-to-a-minute it takes to read the
nested fields further down.

(In practice this was run in a few smaller batches to avoid a single call timing out;
the results below are the real numbers from those runs merged into one complete list —
nothing here is invented, only the presentation is combined.)

In [ ]:
import pyarrow.parquet as pq

pf = pq.ParquetFile('food.parquet')
total = pf.metadata.num_rows
num_rg = pf.metadata.num_row_groups

# all 57 scalar fields
target = ['additives_n','brands','categories','code','compared_to_category','complete',
'completeness','created_t','creator','environmental_score_data','environmental_score_grade',
'environmental_score_score','emb_codes','ingredients_from_palm_oil_n','ingredients_n',
'ingredients_percent_analysis','ingredients_with_specified_percent_n',
'ingredients_with_unspecified_percent_n','ingredients_without_ciqual_codes_n','ingredients',
'known_ingredients_n','labels','lang','last_editor','last_image_t','last_modified_by',
'last_modified_t','last_updated_t','link','manufacturing_places','max_imgid','new_additives_n',
'no_nutrition_data','nova_group','nova_groups','nutriscore_grade','nutriscore_score',
'nutrition_data_per','obsolete','origins','owner','packagings_complete','packaging',
'popularity_key','product_quantity_unit','product_quantity','quantity','rev','scans_n',
'serving_quantity','serving_size','stores','unique_scans_n','unknown_ingredients_n',
'with_non_nutritive_sweeteners','with_sweeteners','schema_version']

name_to_idx = {}
rg0 = pf.metadata.row_group(0)
for i in range(rg0.num_columns):
    name_to_idx[rg0.column(i).path_in_schema] = i

null_counts = {t: 0 for t in target}
for rg in range(num_rg):
    rgmeta = pf.metadata.row_group(rg)
    for t in target:
        idx = name_to_idx[t]
        stats = rgmeta.column(idx).statistics
        if stats is not None and stats.has_null_count:
            null_counts[t] += stats.null_count

results = []
for t in target:
    nn = total - null_counts[t]
    results.append((t, nn, nn / total * 100))
results.sort(key=lambda x: x[2])
for t, nn, pct in results:
    print(f'{t}: {nn:,} ({pct:.1f}%)')

new_additives_n: 237 (0.0%)
with_sweeteners: 4,831 (0.1%)
with_non_nutritive_sweeteners: 31,732 (0.7%)
owner: 112,882 (2.4%)
packagings_complete: 481,936 (10.3%)
no_nutrition_data: 620,035 (13.2%)
packaging: 717,334 (15.3%)
origins: 812,233 (17.3%)
manufacturing_places: 812,980 (17.3%)
emb_codes: 843,454 (17.9%)
link: 887,104 (18.9%)
environmental_score_score: 906,492 (19.3%)
ingredients_from_palm_oil_n: 924,318 (19.7%)
stores: 966,541 (20.6%)
nova_group: 1,152,392 (24.5%)
nova_groups: 1,152,392 (24.5%)
ingredients_percent_analysis: 1,154,691 (24.6%)
product_quantity_unit: 1,265,848 (26.9%)
additives_n: 1,309,161 (27.8%)
ingredients_n: 1,309,174 (27.8%)
ingredients: 1,309,208 (27.8%)
ingredients_without_ciqual_codes_n: 1,309,141 (27.8%)
ingredients_with_specified_percent_n: 1,309,156 (27.8%)
ingredients_with_unspecified_percent_n: 1,309,156 (27.8%)
known_ingredients_n: 1,309,167 (27.8%)
unknown_ingredients_n: 1,309,187 (27.8%)
nutriscore_score: 1,397,325 (29.7%)
product_quantity: 1,433

**Grouping these 57 fields by coverage band**:

- **Near-100% coverage** (`code`, `obsolete`, `rev`, `created_t`, `last_modified_t`,
  `complete`, `lang`, `creator`, `completeness`, `popularity_key`, `last_updated_t`,
  `schema_version`, etc.): these are almost certainly system-generated metadata rather
  than contributor-entered fields, so nearly every row has them. **This is also the
  evidence behind the earlier claim that once `last_modified_t` and `countries_tags`
  are added to the extraction script, their coverage should come out high.**
- **60%+ coverage**: `brands` (65.9%), `max_imgid`/`last_image_t` (75.7%),
  `environmental_score_grade` (82.5%), `last_editor`/`last_modified_by` (97%+),
  `nutriscore_grade` (99.1% — though this doesn't mean "has a real grade"; a large
  share are the placeholder `not-applicable`, consistent with what `DATABASE_GUIDE.md`
  already notes about `products.db`)
- **20%-50% coverage**: the packaging/serving-size family — `product_quantity`,
  `serving_quantity`, `serving_size`, `quantity` — all sit in the 27%-39% range,
  along with `labels` (31.9%) and `nutrition_data_per` (50.4%)
- **Under 20%**: `owner` (2.4%, almost nobody fills this in), and mostly
  supply-chain-detail fields like `packaging`, `origins`, `manufacturing_places`,
  `link`, `stores`
- **Essentially always empty (<1%)**: `new_additives_n` (0.0%), `with_sweeteners`
  (0.1%), `with_non_nutritive_sweeteners` (0.7%) — these are narrow, rarely-assessed
  flags (e.g. "does this contain sweeteners") that almost no product has ever had
  annotated. Not relevant to the current project scope.

This is a **global, 4,701,613-row** profile, not one specific to the baby-food subset —
the natural next step, if we need the true picture for what will actually land in
`products.db`, is to re-run this same profiling filtered down to just the baby-foods
rows rather than assuming the global numbers carry over.

## 3. Key "tag" nested fields: present vs actually non-empty

List-type fields have a trap: a row "having" the field (non-null) and the field
"actually containing something" (a non-empty array) are two different things — e.g.
`allergens_tags` might be non-null on nearly every row while still being an empty list
`[]` most of the time. Checking both readings for a few tag fields that matter later:

In [ ]:
import pyarrow.parquet as pq
import pyarrow.compute as pc
cols = ['categories_tags','labels_tags','allergens_tags','additives_tags','brands_tags']
table = pq.read_table('food.parquet', columns=cols)
total = table.num_rows
for c in cols:
    col = table.column(c)
    n_null = col.null_count
    lens = pc.list_value_length(col)
    non_empty = pc.sum(pc.and_(pc.is_valid(lens), pc.greater(lens, 0))).as_py()
    print(f'{c}: non-null {total-n_null} ({(total-n_null)/total*100:.1f}%), of which actually non-empty (has content) {non_empty} ({non_empty/total*100:.1f}%)')

categories_tags: non-null 2044473 (43.5%), of which actually non-empty (has content) 1887224 (40.1%)
labels_tags: non-null 1716781 (36.5%), of which actually non-empty (has content) 1263731 (26.9%)
allergens_tags: non-null 4658725 (99.1%), of which actually non-empty (has content) 757596 (16.1%)
additives_tags: non-null 1356875 (28.9%), of which actually non-empty (has content) 712440 (15.2%)
brands_tags: non-null 3099024 (65.9%), of which actually non-empty (has content) 2988485 (63.6%)

`allergens_tags` is the textbook example: 99.1% of rows "have" the field, but only
16.1% actually have a real, non-empty allergen list — the system attaches this field's
structure to nearly every row (even when empty), so "non-null coverage" alone can look
much healthier than the data actually is.

## 4. Pulling one real record: what the nested fields actually look like

The type definitions above show `nutriments`, `images`, and `product_name` as list/struct
types, but that's not very intuitive on its own — pulling one real record out makes it
concrete. No "baby foods" filter is doing any real work here conceptually; the intent is
purely to see the data's structure. The `categories_tags` contains `baby-foods` condition
was only used to pick an example relevant to later work, not because this step is driven
by anyone's feedback.

Because heavy nested struct fields like `nutriments`/`images` are relatively slow to
decompress, this uses streaming reads (`iter_batches`) and stops at the first matching
record instead of scanning all 4,701,613 rows.

In [ ]:
import pyarrow.parquet as pq
import json

pf = pq.ParquetFile('food.parquet')
cols = ['code','product_name','brands','quantity','categories_tags','nutriments','images','labels_tags']
found = None
for batch in pf.iter_batches(columns=cols, batch_size=2000):
    cats = batch.column('categories_tags').to_pylist()
    for i, c in enumerate(cats):
        if c and any('baby-foods' in x for x in c):
            found = batch.slice(i, 1).to_pylist()[0]
            break
    if found:
        break

found['nutriments'] = found['nutriments'][:6] if found['nutriments'] else found['nutriments']
found['images'] = found['images'][:2] if found['images'] else found['images']
print(json.dumps(found, indent=2, ensure_ascii=False, default=str))

{
  "code": "0015000045616",
  "product_name": [
    {
      "lang": "main",
      "text": "Grain & grow Puffs Fig Berry"
    },
    {
      "lang": "en",
      "text": "Grain & grow Puffs Fig Berry"
    }
  ],
  "brands": "Gerber",
  "quantity": "42 g",
  "categories_tags": [
    "en:baby-foods"
  ],
  "nutriments": [
    {
      "name": "fruits-vegetables-legumes-estimate-from-ingredients",
      "value": null,
      "100g": 0.5,
      "serving": 0.5,
      "unit": null,
      "prepared_value": null,
      "prepared_100g": null,
      "prepared_serving": null,
      "prepared_unit": null
    },
    {
      "name": "added-sugars",
      "value": 1.0,
      "100g": 1.0,
      "serving": 0.07000000029802322,
      "unit": "g",
      "prepared_value": null,
      "prepared_100g": null,
      "prepared_serving": null,
      "prepared_unit": null
    },
    {
      "name": "carbohydrates",
      "value": 6.0,
      "100g": 6.0,
      "serving": 0.41999998688697815,
      "unit": "g",
     

Barcode `0015000045616` is also already a row in `products.db`.

## Summary: what this EDA pass found

- `food.parquet`: 4,701,613 rows (all OFF products globally), 111 fields — 57 scalar,
  54 nested.
- Scalar field coverage spans a huge range, from under 1% (narrow, rarely-assessed flag
  fields) to near-total coverage (system-generated ID/metadata fields). The
  packaging/serving fields (`product_quantity`, `serving_quantity`, `serving_size`) sit
  consistently around 27%-39% — that's the source data itself being sparse, not an
  extraction-script problem.
- Fields like `allergens_tags` need both a "non-null" and a "non-empty" reading — they
  are not the same number.
- `nutriments` is a name/value list; `images` carries its own `key` field for
  identifying the front image — both are real structural constraints the
  extraction/cleaning script design needs to account for.

**All of the above is the global profile.** It hasn't yet been re-run specifically on
the baby-food subset (the rows that will actually make up `products.db`) — that's the
natural next step.
